![image_1781178167688.png](./image_1781178167688.png "image_1781178167688.png")

**Why Use Frameworks like Langchain and LangGraph?**

- Even though you can write plain Python, frameworks help with things that get complicated fast:

- Workflow Management: Organize multi-step tasks, agents, and APIs cleanly.

- State Management: Keep track of variables, context, or intermediate results across steps.

- Memory: Remember past interactions or user inputs for smarter responses.

- Retries & Error Handling: Automatically retry failed steps or handle exceptions.

- Reusability & Maintainability: Reuse components, swap models/tools without rewriting everything.

- Dynamic Logic & Branching: Easily implement loops, conditional paths, and agent collaboration.


In [0]:
%pip install langgraph

In [0]:
# Restart python kernel 
dbutils.library.restartPython()

In [0]:
# Basic use of Langchain 
from databricks_langchain import ChatDatabricks

# Initialize the model
llm = ChatDatabricks(model="databricks-gpt-oss-120b")


print(llm.invoke("What is the capital of France"))

In [0]:
import json
from databricks_langchain import ChatDatabricks
from langchain_core.messages import HumanMessage, AIMessage

# Initialize model
llm = ChatDatabricks(
    model="databricks-gpt-oss-120b"
)


# Store messages 
messages = []
# User message 
user_msg = HumanMessage(content="What is Databricks?")
messages.append(user_msg)


# AI message / Model response
response = llm.invoke(messages)

# Save reponse 
messages.append( AIMessage(content=response.content))

# Print Messages Accordingly 
print("\n --- All Messages so far ----")
for m in messages:
    print(f"\n[m.type.upper() MESSAGE]")
    if isinstance(m.content, list):
        #Print structured message (reasoning + text)
        for part in m.content:
            print(f" - Type: {part.get('type')} ")
            if "summary" in part:
                print("    Summary:")
                print(json.dumps(part['summary'], indent=4))
            if "text" in part:
                print(f" - Text:\n {part['text'][:1000]} ")
    else:
        print(f"Message Content - {m.content}")


In [0]:
# LangGraph with Databricks LLM - Code Explanation
import os
import random
from typing import Literal, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated

from databricks_langchain import  ChatDatabricks
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

llm = ChatDatabricks(endpoint = "databricks-gpt-oss-120b")

"""In langGraph, MessageState is a built-in helper class (or type) there represents the state of messages flowing through a LangGraph state graph — especially useful when you’re building conversational or agentic workflows where messages (like chat history) are passed between nodes."""

def call_llm(state: MessagesState):
    system_msg = {
        "role": "system",
        "content": "Reply only with plain text. No formatting."
    }

    all_msgs = [system_msg] + state["messages"]

    return {
        "messages": [llm.invoke(all_msgs)]
    }


builder = StateGraph(MessagesState)
builder.add_node("call_llm", call_llm)


builder.add_edge(START, "call_llm")
builder.add_edge("call_llm", END)

graph = builder.compile()

messages = graph.invoke({
    "messages": [HumanMessage("Tell me more about Databricks ?")]
})



     


In [0]:
import json

message = messages['messages'][-1]

# Parse the JSON string into a list
content_list = json.loads(message.content)

for part in content_list:
    if part.get("type") == "text":
        ai_message = part.get("text", "")
        print(ai_message)
        